In [ ]:
clientes = spark.read.csv(
    "../data/clientes.csv",
    header=True,
    inferSchema=True
)

productos = spark.read.csv(
    "../data/productos.csv",
    header=True,
    inferSchema=True
)

ventas = spark.read.csv(
    "../data/ventas.csv",
    header=True,
    inferSchema=True
)

print("Datos cargados correctamente")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("ProyectoVentas") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

In [ ]:
ventas_cliente = ventas.join(
    clientes,
    ventas.id_cliente == clientes.id_cliente,
    "inner"
)
ventas_completo = ventas_cliente.join(
    productos,
    ventas_cliente.id_producto == productos.id_producto,
    "inner"
)   


In [ ]:
ventas_completo.select(
    "nombre",
    "ciudad",
    "producto",
    "categoria",
    "cantidad",
    "precio"
).show()

In [ ]:
v = ventas.alias("v")
c = clientes.alias("c")
p = productos.alias("p")

In [ ]:
ventas.withColumn( "total",
F.col("cantidad") * F.col("precio")
)
ventas.select(
    "id_venta",
    "id_cliente",
    "id_producto",
    "cantidad",
    "precio",
    (F.col("cantidad") * F.col("precio")).alias("total")
).show()

In [ ]:
ventas.withColumn(
    "tipoventa",
    F.when(F.col("cantidad")> 1, "varias unidades").otherwise("una unidad")).select(
    "id_venta",
    "id_cliente",
    "id_producto",
    "cantidad",
    "precio",
    "tipoventa"
).show()

In [ ]:
ventas_total=ventas.withColumn( "total",
F.col("cantidad") * F.col("precio")
)


In [ ]:
ventas_total.groupBy("id_cliente").agg(
    F.sum("total").alias("total_gastado")
).show()

In [ ]:
ventas_total.groupBy("id_cliente").agg(
    F.sum("total").alias("total_gastado"),
    F.count("*").alias("total_compras"),
    F.avg("total").alias("promedio_compra"),
    F.max("total").alias("compra_mas_alta")
).show()


In [ ]:
clientes_limpios = clientes.withColumn(
    "nombres_limpios",
    F.upper(F.trim(F.col("nombre"))))

In [ ]:
clientes_limpios.show()

In [ ]:
clientes_mayusculas = clientes.withColumn(
    "nombre_mayusculas",
    F.upper(F.col("nombre")))
 

In [ ]:
clientes_mayusculas.show()

In [ ]:
cliente_nuevo = clientes.withColumn(
    "cliente_info",
    F.concat_ws(" - ", F.col("nombre"), F.col("ciudad")))

In [ ]:
cliente_nuevo.show()

In [ ]:
clientes_nonull = clientes.filter(
    F.col("ciudad").isNotNull()
)

In [ ]:
cambios_ventas = ventas.withColumn(
    "cantidad", F.col("cantidad").cast("int"))

In [ ]:
cambios_ventas = ventas.withColumn(
    "cantidad", F.col("cantidad").cast("long"))

In [ ]:
cambios_ventas = cambios_ventas.withColumn(
    "precio", F.col("precio").cast("double"))

In [ ]:
cambios_ventas = cambios_ventas.withColumn(
    "id_cliente", F.col("id_cliente").cast("int"),
    )

In [ ]:
cambios_ventas.printSchema

In [ ]:
clientes_cambios = clientes.withColumn(
    "nombre", F.lower(
        F.trim(
            F.col("nombre"))
    )
)
clientes_cambios.show()


In [ ]:
nombre_limpio = clientes.withColumn(
    "nombre", F.lower(F.col("nombre")))

In [ ]:
datos = [
    ("18/08/2026",),
    ("20/08/2026",),
    ("25/12/2026",)
]

df_fechas = spark.createDataFrame(
    datos,
    ["fecha"]
)

In [ ]:
cambio_datos = df_fechas.withColumn(
    "fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy")
)

In [ ]:
df_fechas.printSchema

In [ ]:
cambio_datos.printSchema

In [ ]:
clientes_data = [
    (1, "  Mostafa  ", "Hmidi", "Madrid"),
    (2, "JUAN", "Perez", "Barcelona"),
    (3, "Ana", "Garcia", "Madrid"),
    (4, "Luis", "Lopez", None)
]

clientes2 = spark.createDataFrame(
    clientes_data,
    ["id_cliente", "nombre", "apellido", "ciudad"]
)

In [ ]:
ventas_data = [
    (1, 101, "2", "15.50", "18/08/2026"),
    (1, 102, "1", "30.00", "19/08/2026"),
    (2, 101, "5", "15.50", "20/08/2026"),
    (3, 103, "3", "50.00", "21/08/2026"),
    (4, 102, "1", "30.00", "25/08/2026")
]

ventas2 = spark.createDataFrame(
    ventas_data,
    ["id_cliente", "id_producto", "cantidad", "precio", "fecha"]
)

In [ ]:
clientes2.show()

In [ ]:
clientes2.select(
"nombre","ciudad"
).show()

In [ ]:
clientes2.filter(
    F.col("ciudad") == "Madrid"
).show()

In [ ]:
clientes2.filter(
    F.col("ciudad").isNotNull()
).show()

In [ ]:
clientes2_limpio = clientes2.withColumn(
    "nombre_limpio", F.lower(F.trim(F.col("nombre")))
)


In [ ]:
clientes2_limpio.show()

In [ ]:
clientes2_unidos = clientes2_limpio.withColumn(
    "nombre_completo",
    F.concat_ws(
        " ",
        F.col("nombre_limpio"),
        F.col("apellido")
    )
)
clientes2_unidos.show()

In [ ]:
clientes2_cst= clientes2.withColumn(
    "id_cliente", F.col("id_cliente").cast("string")
)

In [ ]:
ventas = ventas.withColumn(
    "cantidad",
    F.col("cantidad").cast("long")
).withColumn(
    "precio",
    F.col("precio").cast("double")
)

In [ ]:
ventas_clientes = ventas.join(
    clientes,
    ventas.id_cliente == clientes.id_cliente,
    "inner"
)


resultado = ventas_clientes.groupBy(
    "nombre",
    "ciudad",
    "id_cliente"
).agg(
    F.sum(
        F.col("cantidad") * F.col("precio")
    ).alias("total_gastado")
)

resultado = resultado.orderBy(
    F.col("total_gastado").desc()
)

resultado = resultado.select(
    "nombre",
    "ciudad",
    "total_gastado"
)

resultado.show()

In [ ]:
ventas.printSchema()

In [ ]:
ventas_clientes.printSchema()

In [ ]:
ventas_clientes.select(
    "cantidad",
    "precio"
).show()

In [ ]:
ventas_clientes.groupBy(
    "nombre",
    "ciudad",
    "id_cliente"
).sum("precio").show()

In [ ]:
ventas_clientes = ventas.join(
    clientes,
    "id_cliente",
    "inner"
)

resultado = ventas_clientes.groupBy(
    "id_cliente",
    "nombre",
    "ciudad"
).agg(
    F.sum(
        F.col("cantidad") * F.col("precio")
    ).alias("total_gastado")
)

resultado = resultado.orderBy(
    F.col("total_gastado").desc()
)

resultado.select(
    "nombre",
    "ciudad",
    "total_gastado"
).show()

In [ ]:
ventas = ventas.withColumn(
    "fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy")
)
ventas = ventas.withColumn(
    "año", F.year(F.col("fecha"))
)
ventas = ventas.withColumn(
    "mes", F.month(F.col("fecha"))
)
ventas = ventas.groupBy(
    "año",
    "mes"
).agg(
    F.sum(
        F.col("cantidad") * F.col("precio")
    ).alias("total_ventas")
).orderBy(
    "año",
    "mes"
)
ventas.show()

In [ ]:
ventas_clientes = ventas.join(
    clientes,
    "id_cliente",
    "inner" 
)
ventas_clientes = ventas_clientes.fillna(
    {
        "ciudad": "Desconocida"
    }
)
ventas_por_ciudad = ventas_clientes.groupBy(
    "ciudad"
).agg(
    F.sum(
        F.col("cantidad") * F.col("precio")
    ).alias("total_ventas"),
    F.count("*").alias("total_transacciones")
)
ventas_por_ciudad = ventas_por_ciudad.orderBy(
    F.col("total_ventas").desc()
)
ventas_por_ciudad.show()

In [ ]:
ventas = ventas.withColumn(
    "cantidad", F.col("cantidad").cast("long")
).withColumn(
    "precio", F.col("precio").cast("double")
).withColumn(
    "total", F.col("cantidad") * F.col("precio")
)
ventas_clientes = ventas.join(
    clientes,
    "id_cliente",
    "inner"
)
ventas_clientes = ventas_clientes.groupBy(
    "id_cliente",
    "nombre",
    "ciudad"
).agg(
    F.sum("total").alias("total_gastado"),
        F.avg("total").alias("promedio"),
        F.count("*").alias("numero_ventas"),
        F.sum("cantidad").alias("unidades_totales"),
        F.countDistinct("id_producto").alias("productos_diferentes")
)
resultado = ventas_clientes.withColumn(
    "categoria",
    F.when(
        F.col("total_gastado") >= 100, "premium"
    ).when(
    F.col("total_gastado") >= 50 , "medio"
).otherwise("normal"))
resultado = resultado.orderBy(
    F.col("total_gastado").desc()
)
resultado.show()



In [6]:
ventas_clientes = ventas.join(
    clientes,
    "id_cliente",
    "inner"
)

ventas_clientes = ventas_clientes.fillna(
    {
        "ciudad": "Desconocida"
    }
)

ventas_clientes = ventas_clientes.withColumn(
    "total",
    F.col("cantidad") * F.col("precio")
)

resultado = ventas_clientes.groupBy(
    "ciudad"
).agg(
    F.countDistinct("id_cliente").alias("numero_clientes"),
    F.count("*").alias("numero_ventas"),
    F.sum("cantidad").alias("unidades_vendidas"),
    F.sum("total").alias("total_gastado"),
    F.avg("precio").alias("precio_medio")
)

resultado = resultado.withColumn(
    "ciudad_rentable",
    F.when(
        F.col("total_gastado") >= 100,
        "alta"
    ).when(
        F.col("total_gastado") >= 50,
        "media"
    ).otherwise(
        "baja"
    )
)

resultado = resultado.orderBy(
    F.col("total_gastado").desc()
)

resultado.show()

+---------+---------------+-------------+-----------------+-------------+------------+---------------+
|   ciudad|numero_clientes|numero_ventas|unidades_vendidas|total_gastado|precio_medio|ciudad_rentable|
+---------+---------------+-------------+-----------------+-------------+------------+---------------+
|   Madrid|              1|            2|                3|         1300|       550.0|           alta|
|Barcelona|              1|            1|                2|           50|        25.0|          media|
| Valencia|              1|            1|                1|           50|        50.0|          media|
|  Sevilla|              1|            1|                1|           25|        25.0|           baja|
+---------+---------------+-------------+-----------------+-------------+------------+---------------+



In [12]:
ventas = ventas.withColumn(
    "cantidad",
    F.col("cantidad").cast("long")
).withColumn(
    "precio",
    F.col("precio").cast("double")
).withColumn(
    "total",
    F.col("cantidad") * F.col("precio")
)

resultado = ventas.groupBy(
    "id_producto"
).agg(
    F.count("*").alias("numero_ventas"),
    F.sum("cantidad").alias("unidades_vendidas"),
    F.sum("total").alias("facturacion_total"),
    F.avg("precio").alias("precio_medio")
)

resultado = resultado.withColumn(
    "tipo_producto",
    F.when(
        F.col("facturacion_total") >= 100,
        "top"
    ).when(
        F.col("facturacion_total") >= 50,
        "medio"
    ).otherwise(
        "bajo"
    )
)

resultado = resultado.orderBy(
    F.col("facturacion_total").desc()
)

resultado.show()

+-----------+-------------+-----------------+-----------------+------------+-------------+
|id_producto|numero_ventas|unidades_vendidas|facturacion_total|precio_medio|tipo_producto|
+-----------+-------------+-----------------+-----------------+------------+-------------+
|        101|            1|                1|            900.0|       900.0|          top|
|        104|            1|                2|            400.0|       200.0|          top|
|        102|            2|                3|             75.0|        25.0|        medio|
|        103|            1|                1|             50.0|        50.0|        medio|
+-----------+-------------+-----------------+-----------------+------------+-------------+

